# Can outlier ranking find planted label mistakes?


**A synthetic task, stated first.** 5 % of the genes with a value in each label have that value swapped to another class of the same label (seed 0), in a copy of the table. Label-outlier ranking runs on the corrupted label; the planted genes are counted among its top calls at depths fixed here (50, 100, 200). This says whether the method finds mistakes we made; it says nothing about which real annotations are wrong (instruction 64.12).

Built 2026-11-07.

In [1]:
import os, numpy as np, pandas as pd
from parascape import strategies as S
base = S.Context.shipped('Tg')
LABELS = ['compartment', 'dtm_class', 'cellcycle_phase']
rows = []
for label in LABELS:
    nodes = base.nodes.copy()
    values = nodes[label]
    known = values.dropna().index
    rng = np.random.default_rng(0)
    planted = rng.choice(known, size=max(1, int(0.05 * len(known))), replace=False)
    classes = values.dropna().unique()
    for i in planted:
        nodes.at[i, label] = rng.choice([c for c in classes if c != values.at[i]])
    ctx = S.Context(nodes, organism='Tg')
    r = S.get('label_outliers').run(ctx, target=label, top=200)
    table = next(t for t in r.tables.values() if 'gene_id' in getattr(t, 'columns', []))
    ranked = table.gene_id.astype(str).tolist()
    planted_ids = set(nodes.loc[planted, 'gene_id'].astype(str))
    hits = np.isin(ranked, list(planted_ids))
    prevalence = len(planted_ids) / len(known)
    row = dict(label=label, labelled=len(known), planted=len(planted_ids), prevalence=round(prevalence, 3))
    for k in (50, 100, 200):
        row[f'precision_at_{k}'] = round(float(hits[:k].mean()), 3) if len(hits) >= k else np.nan
        row[f'recall_at_{k}'] = round(float(hits[:k].sum() / len(planted_ids)), 3)
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT, 'planted_recovery.tsv'), sep='\t', index=False)
table

,label,labelled,planted,prevalence,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_200,recall_at_200
0,compartment,8140,407,0.05,0.42,0.052,0.49,0.12,0.515,0.253
1,dtm_class,8140,407,0.05,0.74,0.091,0.72,0.177,0.73,0.359
2,cellcycle_phase,873,43,0.049,0.32,0.372,0.22,0.512,0.14,0.651
